## WARNING
When initializing environment, kamodo **WILL NOT WORK** unless you specifically open the conda environment "Komodo_env".

Komodo runs off Python 3.11 and as such requires various older versions of common libraries. An environment is *required* to initialize Komodo.

## Data Functionalization

In [3]:
from kamodo_ccmc.tools.functionalize import Functionalize_Dataset
help(Functionalize_Dataset)

Help on function Functionalize_Dataset in module kamodo_ccmc.tools.functionalize:

Functionalize_Dataset(coord_dict, data_dict, kamodo_object=None, coord_str='', func=None, func_default='data')
    Determine and call the correct functionalize routine.
    Inputs:
        coord_dict: a dictionary containing the coordinate information.
            {'name_of_coord1': {'units': 'coord1_units', 'data': coord1_data},
             'name_of_coord2': {'units': 'coord2_units', 'data': coord2_data},
             etc...}
            coordX_data should be a 1D array. All others should be strings.
        data_dict: a dictionary containing the data information.
            {'variable_name1': {'units': 'data1_units', 'data': data1_array},
             'variable_name2': {'units': 'data2_units', 'data': data2_array},
             etc...}
            dataX_array should have the same shape as
                (coord1, coord2, coord3, ..., coordN)
        Note:The datasets given in the data_dict dictionary

In [2]:
# Example of functionalizing as a 7D array.
# Any number of dimensions can be specified.

import numpy as np
rng1 = np.random.RandomState(1) # Seed random number generators differently
rng2 = np.random.RandomState(2) # To prevent identical overlap
# Initialize coordinate dictionary
coord_dict: str = {
            'time': {'units': 'hr', 'data': np.linspace(0., 24., 25)},
            'lon': {'units': 'deg', 'data': np.linspace(-180., 180., 12)}, # longitude
            'lat': {'units': 'deg', 'data': np.linspace(-90., 90., 5)},    # latitude
            'radius': {'units': 'R_E', 'data': np.linspace(0., 50., 10)},
            'nonsense': {'units': 'm/m', 'data': np.linspace(1., 15., 15)},
            'nope': {'units': 'm', 'data': np.linspace(1., 150., 25)},
            'nada': {'units': 'hPa', 'data': np.linspace(0.00005, 15000., 20)}
              }
# Initialize variable dictionary
var_dict = {
            'Test_7D': {'units': 'S', 'data': rng1.rand(25, 12, 5, 10, 15, 25, 20)},
            'Good_7D': {'units':'mK','data': rng2.rand(25, 12, 5, 10, 15, 25, 20)}
}
kamodo_object = Functionalize_Dataset(coord_dict, var_dict)
kamodo_object

{Test_7D(time, lon, lat, radius, nonsense, nope, nada): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE06FC0>, Test_7D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE06FC0>, Good_7D(time, lon, lat, radius, nonsense, nope, nada): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37ABA2C00>, Good_7D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37ABA2C00>}

#### Plotting

In [ ]:
# Generating a 1D plot
# Utilizing previously established array

kamodo_object.plot('Test_7D', 'Good_7D', plot_partial={
    'Test_7D': {'time': 12., 'lon': 0.5, 'lat': -20., 'radius': 15., 'nonsense': 11.5, 'nope': 5.},
    'Good_7D': {'time': 12., 'lon': 0.5, 'lat': -20., 'radius': 15., 'nonsense': 11.5, 'nope': 5.}
})

In [ ]:
# Generating a 2D plot

kamodo_object.plot('Test_7D', plot_partial={
    'Test_7D': {'time':12.,'lon':0.5,'lat':-20.,'radius':15.,'nonsense':11.5}
})

In [3]:
#Adding new Functionalized Datasets to a Kamodo Object

coord_dict = {
    'time': {'units': 'hr', 'data': np.linspace(0., 24., 25)}
    }
var_dict = {
    'Test_1D': {'units': 'S', 'data': rng1.rand(25)},
            'Good_1D': {'units': 'mK', 'data': rng2.rand(25)}
    }
kamodo_object = Functionalize_Dataset(coord_dict, var_dict, kamodo_object)
kamodo_object

{Test_7D(time, lon, lat, radius, nonsense, nope, nada): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE06FC0>, Test_7D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE06FC0>, Good_7D(time, lon, lat, radius, nonsense, nope, nada): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37ABA2C00>, Good_7D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37ABA2C00>, Test_1D(time): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE07D80>, Test_1D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE07D80>, Good_1D(time): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E3E74702C0>, Good_1D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E3E74702C0>}

In [4]:
# You can plot all functions on the same plot
# as long as the independent variable is the same

kamodo_object.plot('Test_1D', 'Good_1D', 'Test_7D', 'Good_7D', plot_partial={
    'Test_1D': {},
    'Good_1D': {},
    'Test_7D': {'lon': 0.5, 'lat': -20., 'radius': 15., 'nonsense': 11.5, 'nope': 5., 'nada': 12.},
    'Good_7D': {'lon': 0.5, 'lat': -20., 'radius': 15., 'nonsense': 11.5, 'nope': 5., 'nada': 12.}})

# The tutorial does not include the partials for Test_1D and Good_1D.
# 1D and 7D are functions of one and seven variables respectively.
# Including 1D in partial structure unifies plotting space on
# the time axis.

In [5]:
# Interpolation: Estimation of unknown values between known values.

# You even use a custom interpolator if desired for a new dataset added to the same kamodo_object.
# The interpolator must be defined separately for each dataset.
coord_dict = {'time': {'units': 'hr', 'data': np.linspace(0., 24., 25)},
              'lon': {'units': 'deg', 'data': np.linspace(-180., 180., 12)},
              'lat': {'units': 'deg', 'data': np.linspace(-90., 90., 5)}}
var_dict = {'TestCustomA_3D': {'units': 'S', 'data': rng1.rand(25, 12, 5)},
            'TestCustomB_3D': {'units': 'm/s', 'data': rng2.rand(25, 12, 5)*-2.}}

# Define a custom interpolator (simple example)
# see https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.RegularGridInterpolator.html
from numpy import NaN
from scipy.interpolate import RegularGridInterpolator as RGI
coord_list = [value['data'] for key, value in coord_dict.items()]
for key in var_dict.keys():
    rgi = RGI(coord_list, var_dict[key]['data'], bounds_error=False,
                fill_value=-10., method='nearest')
    # wrap in a function and return the function
    def interp(xvec):
        return rgi(xvec)
    tmp_dict = {key: var_dict[key]}  # construct a separate dictionary for the current variable
    kamodo_object = Functionalize_Dataset(coord_dict, tmp_dict, kamodo_object, func=interp, func_default='custom')
kamodo_object

{Test_7D(time, lon, lat, radius, nonsense, nope, nada): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE06FC0>, Test_7D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE06FC0>, Good_7D(time, lon, lat, radius, nonsense, nope, nada): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37ABA2C00>, Good_7D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37ABA2C00>, Test_1D(time): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE07D80>, Test_1D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E37BE07D80>, Good_1D(time): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E3E74702C0>, Good_1D: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E3E74702C0>, TestCustomA_3D(time, lon, lat): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E3E9CDF9C0>, TestCustomA_3D: <

In [ ]:
# Not entirely sure why, but this plotting doesn't work.
'''
kamodo_object.plot('TestCustomB_3D', plot_partial={
    'TestCustomB_3D': {'time': 12.58}
})
'''

#### Metadata Functions

In [6]:
# Access metadata
kamodo_object['Test_1D'].meta

{'units': 'S',
 'arg_units': {'time': 'hr'},
 'citation': None,
 'equation': None,
 'hidden_args': []}

In [7]:
# Add to metadata
kamodo_object['Test_1D'].meta['description'] = 'Testing the functionalize.py script'
kamodo_object['Test_1D'].meta['citation'] = 'Sendgikoski et al. 2026'
kamodo_object['Test_1D'].meta

{'units': 'S',
 'arg_units': {'time': 'hr'},
 'citation': 'Sendgikoski et al. 2026',
 'equation': None,
 'hidden_args': [],
 'description': 'Testing the functionalize.py script'}

In [10]:
# Visualize pandas format output
kamodo_object.detail()

,symbol,units,lhs,rhs,arg_units
Test_7D,"Test_7D(time, lon, lat, radius, nonsense, nope...",S,Test_7D,"lambda(time, lon, lat, radius, nonsense, nope,...","{'time': 'hr', 'lon': 'deg', 'lat': 'deg', 'ra..."
Good_7D,"Good_7D(time, lon, lat, radius, nonsense, nope...",mK,Good_7D,"lambda(time, lon, lat, radius, nonsense, nope,...","{'time': 'hr', 'lon': 'deg', 'lat': 'deg', 'ra..."
Test_1D,Test_1D(time),S,Test_1D,lambda(time),{'time': 'hr'}
Good_1D,Good_1D(time),mK,Good_1D,lambda(time),{'time': 'hr'}
TestCustomA_3D,"TestCustomA_3D(time, lon, lat)",S,TestCustomA_3D,"lambda(time, lon, lat)","{'time': 'hr', 'lon': 'deg', 'lat': 'deg'}"
TestCustomB_3D,"TestCustomB_3D(time, lon, lat)",m/s,TestCustomB_3D,"lambda(time, lon, lat)","{'time': 'hr', 'lon': 'deg', 'lat': 'deg'}"


In [12]:
# Determine dependent coordinates and ranges
import kamodo_ccmc.flythrough.model_wrapper as MW
MW.Coord_Range(kamodo_object, ['Test_7D'])

The minimum and maximum values for each variable and coordinate are:
Test_7D:
time: [0.0, 24.0, 'hr']
lon: [-180.0, 180.0, 'deg']
lat: [-90.0, 90.0, 'deg']
radius: [0.0, 50.0, 'R_E']
nonsense: [1.0, 15.0, 'm/m']
nope: [1.0, 150.0, 'm']
nada: [5e-05, 15000.0, 'hPa']


## Functionalizing HAPI Results

In [2]:
# Retrieve data and metadata objects from HAPI

# Copied from a https://hapi-server.org/servers/ example python script
from hapiclient import hapi
server     = 'http://planet.physics.uiowa.edu/das/das2Server/hapi'
dataset    = 'Cassini/RPWS/Survey_KeyParam,B'
parameters = 'magnetic_specdens'
start      = '2011-03-27T00:18:00Z'
stop       = '2011-03-28T00:00:00.000Z'
data, meta = hapi(server, dataset, parameters, start, stop)

In [3]:
# import os
# print(os.popen('pip install hapiplot --upgrade').read())

# from hapiplot import hapiplot
# hapiplot(data, meta)

In [4]:
# Functionalize HAPI Outputs with Kamodo

from kamodo_ccmc.tools.functionalize_hapi import functionalize_hapi, varlist
kamodo_object = functionalize_hapi(data, meta)
kamodo_object

{magnetic_specdens(UTC_time, yTags): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E26B5911C0>, magnetic_specdens: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x000001E26B5911C0>}

In [5]:
# Print variable names functionalized
var_list = varlist(meta)
print(var_list)

['magnetic_specdens']


These plotting cells take a moment to load.

In [1]:
# from kamodo_ccmc.tools.plotfunctions import toColor, toLog10
# toLog10(
#     toColor(kamodo_object.plot(var_list[0]), colorscale='Viridis')
#     )

In [2]:
# kamodo_object.plot(var_list[0], plot_partial={
#     var_list[0]: {'UTC_time': 1.301241e9}
# }).update_yaxes(type='log').update_xaxes(type='log')
# add .update_xaxes(type='log') to change x scale to log

### Interpolation with Kamodo

In [ ]:
# Interpolation for a single time value in the range of the dataset.

# create a utc timestamp in date range of retrieved dataset
from datetime import datetime, timezone
sample_datetime = datetime(2011, 3,27,23,50, tzinfo=timezone.utc).timestamp()
print('Sample Timestamp: ', sample_datetime)

kamodo_object[var_list[0]](UTC_time=sample_datetime)

Sample Timestamp:  1301269800.0


array([1.03300e+00, 5.23350e-04, 7.05295e-03, 1.24995e-04, 6.97350e-05,
       2.28880e-05, 1.39670e-05, 8.58450e-06, 1.60700e-05, 3.29600e-05,
       1.74250e-06, 3.54800e-06, 2.49700e-05, 1.50850e-06, 7.99400e-07,
       2.53650e-07, 2.19350e-07, 1.17250e-07, 4.92650e-08, 1.12950e-07,
       5.38900e-07, 2.22050e-08, 1.24000e-08, 6.17200e-09, 3.33650e-09,
       1.41705e-08, 4.67550e-09, 2.13810e-09, 1.39650e-09, 1.88450e-09,
       1.52000e-09, 1.09800e-09, 2.03700e-09, 2.29500e-09, 1.37900e-09,
       1.44800e-09, 1.48350e-09, 1.91600e-09, 2.16300e-09, 2.50900e-09,
       3.45650e-09, 4.29300e-09])

In [17]:
# Create a time grid covering 1 hour, 15min resolution

time_grid = linspace(sample_datetime, sample_datetime+3600, 4, endpoint=True)

kamodo_object[var_list[0]](UTC_time=time_grid)

array([[1.033000e+00, 2.612675e-01, 5.801000e-03, 3.657000e-03],
       [5.233500e-04, 1.754000e-04, 8.303000e-04, 4.588500e-04],
       [7.052950e-03, 2.251000e-04, 4.913000e-04, 5.347500e-04],
       [1.249950e-04, 1.091950e-04, 5.722000e-05, 1.023000e-04],
       [6.973500e-05, 6.437500e-05, 7.490500e-05, 2.886500e-05],
       [2.288800e-05, 1.686500e-05, 9.832500e-06, 2.072000e-05],
       [1.396700e-05, 9.054000e-06, 9.224500e-06, 5.733500e-06],
       [8.584500e-06, 3.220000e-06, 5.858000e-06, 2.697500e-06],
       [1.607000e-05, 1.617500e-05, 1.202000e-05, 1.669000e-05],
       [3.296000e-05, 1.893500e-05, 1.912500e-05, 2.320000e-05],
       [1.742500e-06, 1.220100e-06, 1.385500e-06, 8.027500e-07],
       [3.548000e-06, 2.361500e-06, 1.073900e-05, 8.881500e-06],
       [2.497000e-05, 6.372000e-06, 7.910000e-06, 7.640000e-06],
       [1.508500e-06, 4.122000e-06, 1.682500e-06, 1.340500e-06],
       [7.994000e-07, 2.874500e-06, 1.778000e-06, 1.843000e-06],
       [2.536500e-07, 5.4

In [19]:
# Create a new frequency grid with same range as data
# Retrieve coord grid and determine freq. range
from kamodo import get_defaults
defaults = get_defaults(kamodo_object[var_list[0]])
# coords returned in py dict, coord names as keys
print(defaults)
print(defaults['yTags'].min(), defaults['yTags'].max())

{'UTC_time': array([1.30120671e+09, 1.30120677e+09, 1.30120683e+09, ...,
       1.30129185e+09, 1.30129191e+09, 1.30129197e+09]), 'yTags': array([1.00e+00, 1.26e+00, 1.58e+00, 2.00e+00, 2.51e+00, 3.16e+00,
       3.98e+00, 5.01e+00, 6.31e+00, 7.94e+00, 1.00e+01, 1.26e+01,
       1.58e+01, 1.99e+01, 2.51e+01, 3.16e+01, 3.98e+01, 5.01e+01,
       6.31e+01, 7.94e+01, 1.00e+02, 1.26e+02, 1.58e+02, 2.00e+02,
       2.51e+02, 3.16e+02, 3.98e+02, 5.01e+02, 6.31e+02, 7.94e+02,
       1.00e+03, 1.26e+03, 1.58e+03, 2.00e+03, 2.51e+03, 3.16e+03,
       3.98e+03, 5.01e+03, 6.31e+03, 7.94e+03, 1.00e+04, 1.26e+04])}
1.0 12600.0


In [20]:
# Create new frequency grid and interpolate data onto
# new grid for single timestamp

freq_grid = linspace(
    defaults['yTags'].min(), defaults['yTags'].max(), 10, endpoint=True
)
kamodo_object[var_list[0]](UTC_time = sample_datetime, yTags=freq_grid)

array([1.03300000e+00, 1.51142083e-09, 1.40986718e-09, 1.57615858e-09,
       2.02820556e-09, 2.30956060e-09, 2.72073099e-09, 3.36461192e-09,
       3.84261267e-09, 4.29300000e-09])

In [22]:
# Interpolate onto time grid and frequency grid simulatneously
# Result too large to print. Shape and sample printed.

data = kamodo_object[var_list[0]](UTC_time=time_grid, yTags=freq_grid).T
data.shape, data[0]

((4, 10),
 array([1.03300000e+00, 1.51142083e-09, 1.40986718e-09, 1.57615858e-09,
        2.02820556e-09, 2.30956060e-09, 2.72073099e-09, 3.36461192e-09,
        3.84261267e-09, 4.29300000e-09]))

In [24]:
# Interpolate for new grids of times and frequences:
time_grid = linspace(
    defaults['UTC_time'][0], defaults['UTC_time'][100], 1000, endpoint=True
)
data = kamodo_object[var_list[0]](UTC_time = time_grid, yTags=freq_grid).T
data.shape, data[0]

((1000, 10),
 array([4.03500000e-03, 2.15904306e-09, 1.32031590e-09, 1.48926861e-09,
        1.85222949e-09, 2.07073511e-09, 2.51681456e-09, 3.32820647e-09,
        3.75695205e-09, 4.12200000e-09]))

### Including a custom interpolartor

In [29]:
# LIBRARY BUG: Tuple index out of range
# Not sure how to fix this.

# Retrieve two 1D variables from HAPI and generate plots

# Copied from https://hapi-server.org/servers/#server=SSCWeb&dataset=aee&parameters=X_GEO&start=1975-11-20T21:04:00.000Z&stop=1975-11-22T21:04:00.000Z&return=script&format=python
from hapiclient import hapi
server1     = 'https://iswa.gsfc.nasa.gov/IswaSystemWebApp/hapi'  # 1D example
dataset1    = 'ENLIL_KP_P7M'
parameters1 = 'KP_18,KP_90'  # A comma-separated list of variable names
start1 = '2011-03-27T00:00:00Z'
stop1  = '2011-03-28T00:00:00Z'
data1, meta1 = hapi(server1, dataset1, parameters1, start1, stop1, format='csv')

IndexError: tuple index out of range

## Choosing Models and Variables

In [1]:
# Show list of models currently available

import kamodo_ccmc.flythrough.model_wrapper as MW
MW.Choose_Model('CTIPe')

<module 'kamodo_ccmc.readers.ctipe_4D' from 'c:\\Users\\mageb\\miniconda3\\envs\\Kamodo_env\\Lib\\site-packages\\kamodo_ccmc\\readers\\ctipe_4D.py'>

In [2]:
# Access Documentation

help(MW.Variable_Search)

Help on function Variable_Search in module kamodo_ccmc.flythrough.model_wrapper:

Variable_Search(search_string='', model='', file_dir='', return_dict=False)
    Search variable descriptions for the given string. If the model string
    is set, the chosen model will be searched. If file_dir is set to the
    directory where some model data is stored, the files available in that
    model data will be searched. If neither is set, then all model variables
    will be searched. Searching all of the model variables will take additional
    time as the model library grows. All search strings are converted to lower
    case text before search (e.g. "temperature" not "Temperature").
    
    Returns a dictionary with information that varies per call type if 
    return_dict is True. Default is to print the information to the screen
    and return nothing (return_dict=False).
    - If the search string, model and file directory are all not given or are
        all empty strings, then no dictio

In [4]:
# Show all possible variables for every Model

MW.Variable_Search()

Printing all possible variables across all models...

The ADELPHI model accepts the standardized variable names listed below.
-----------------------------------------------------------------------------------
E_east : '['electric field in eastern direction (increasing longitude) ', 3, 'SM', 'sph', ['time', 'lon', 'lat'], 'mV/m']'
E_north : '['electric field in north direction (increasing latitude) ', 4, 'SM', 'sph', ['time', 'lon', 'lat'], 'mV/m']'
J_Rin : '['Input radial current', 9, 'SM', 'sph', ['time', 'lon', 'lat'], 'mA/m**2']'
J_Rout : '['model-generated radial current would be identical to J_Rin if the model were perfect', 10, 'SM', 'sph', ['time', 'lon', 'lat'], 'mA/m**2']'
J_east : '['electric current in eastern direction(increasing longitude) (height integrated current density)', 5, 'SM', 'sph', ['time', 'lon', 'lat'], 'A/m']'
J_heat : '['Joule heating rate ', 8, 'SM', 'sph', ['time', 'lon', 'lat'], 'mW/m**2']'
J_north : '['electric current in north direction (increasing lat

In [5]:
# Choosing a model by searching for an associated variable

MW.Variable_Search('temperature')


ADELPHI:
No temperature variables found.

AMGeO:
No temperature variables found.

CTIPe:
T_ilev1: ['temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'ilev1'], 'K']
T: ['temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_e: ['electron temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_i: ['ion temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_n_ilev: ['neutral temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'ilev'], 'K']
T_n: ['neutral temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']

DTM:
T_exo: ['Exospheric temperature', 'GDZ-sph', ['time', 'lon', 'lat'], 'K']
T: ['temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']

GAMERA_GM:
No temperature variables found.

GITM:
T_n: ['neutral temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_e: ['electron temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_i: ['ion temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']

IRI:
T_e: ['elec

In [6]:
# Temperature varibles
MW.Variable_Search('temperature', model='CTIPe')

T_ilev1: ['temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'ilev1'], 'K']
T: ['temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_e: ['electron temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_i: ['ion temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_n_ilev: ['neutral temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'ilev'], 'K']
T_n: ['neutral temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']


In [7]:
# ALL variable search
MW.Variable_Search('', model='CTIPe')

rho_ilev1: ['total mass density', 0, 'GDZ', 'sph', ['time', 'lon', 'lat', 'ilev1'], 'kg/m**3']
rho: ['total mass density', 0, 'GDZ', 'sph', ['time', 'lon', 'lat', 'height'], 'kg/m**3']
T_ilev1: ['temperature', 1, 'GDZ', 'sph', ['time', 'lon', 'lat', 'ilev1'], 'K']
T: ['temperature', 1, 'GDZ', 'sph', ['time', 'lon', 'lat', 'height'], 'K']
T_e: ['electron temperature', 2, 'GDZ', 'sph', ['time', 'lon', 'lat', 'height'], 'K']
T_i: ['ion temperature', 3, 'GDZ', 'sph', ['time', 'lon', 'lat', 'height'], 'K']
H_ilev: ['height dependent on primary pressure level', 4, 'GDZ', 'sph', ['time', 'lon', 'lat', 'ilev'], 'm']
H_ilev1: ['height dependent on secondary pressure level', 5, 'GDZ', 'sph', ['time', 'lon', 'lat', 'ilev1'], 'm']
v_nnorth_ilev: ['meridional neutral wind velocity (north)', 6, 'GDZ', 'sph', ['time', 'lon', 'lat', 'ilev'], 'm/s']
v_nnorth: ['meridional neutral wind velocity (north)', 6, 'GDZ', 'sph', ['time', 'lon', 'lat', 'height'], 'm/s']
v_neast_ilev: ['zonal neutral wind velocit

In [11]:
# Choosing a dataset
file_dir = "C:/Users/mageb/OneDrive/Documents/Python_Project/BSA_REU2026/data/CTIPe/3523_Katelynn_Greer_042326_IT_1_raw/out"
# path to data
MW.Variable_Search('', model='CTIPe', file_dir=file_dir)

IndexError: list index out of range

In [13]:
import os
import xarray as xr
from kamodo_ccmc.tools.functionalize import Functionalize_Dataset

file_dir = "C:/Users/mageb/OneDrive/Documents/Python_Project/BSA_REU2026/data/CTIPe/3523_Katelynn_Greer_042326_IT_1_raw/out"

# 1. Load one of your pre-processed variable files directly
# Let's use the density file as an example
density_file = os.path.join(file_dir, "2022-01-12-plot-density.nc")
ds = xr.open_dataset(density_file)

# 2. Inspect the dataset coordinates and variables to see what's inside
print("Coordinates:", list(ds.coords))
print("Variables:", list(ds.data_vars))

Coordinates: ['time', 'plev', 'lat', 'lon']
Variables: ['doy', 'min', 'density', 'height', 'temperature', 'mean_molecular_mass']


In [ ]:
from kamodo_ccmc.tools.functionalize import Functionalize_Dataset
from kamodo import Kamodo
# 1. Structure the coordinate dictionary from xarray arrays
coord_dict = {
    "time": {"units": "hr", "data": ds["time"].values},
    "plev": {"units": "hPa", "data": ds["plev"].values},
    "lat": {"units": "deg", "data": ds["lat"].values},
    "lon": {"units": "deg", "data": ds["lon"].values},
}

# 2. Structure the variable dictionary for the data fields you want
var_dict = {
    "density": {"units": "kg/m^3", "data": ds["density"].values},
    "temperature": {"units": "K", "data": ds["temperature"].values},
    "height": {"units": "km", "data": ds["height"].values},
}

# 3. Initialize your new Kamodo container
kamodo_ctipe = Kamodo()
kamodo_ctipe = Functionalize_Dataset(coord_dict, var_dict, kamodo_ctipe)

# 4. Verify it registered beautifully
kamodo_ctipe.detail()

,symbol,units,lhs,rhs,arg_units
density,"density(time, plev, lat, lon)",kg/m^3,density,"lambda(time, plev, lat, lon)","{'time': 'hr', 'plev': 'hPa', 'lat': 'deg', 'l..."
temperature,"temperature(time, plev, lat, lon)",K,temperature,"lambda(time, plev, lat, lon)","{'time': 'hr', 'plev': 'hPa', 'lat': 'deg', 'l..."
height,"height(time, plev, lat, lon)",km,height,"lambda(time, plev, lat, lon)","{'time': 'hr', 'plev': 'hPa', 'lat': 'deg', 'l..."


In [15]:
# Temperature variables in chosen data
MW.Variable_Search('temperature', model='CTIPe', file_dir=file_dir)

IndexError: list index out of range

In [7]:
# Query dataset

MW.File_Times('CTIPe', file_dir)

IndexError: list index out of range

## Functionalizing a Modeled Dataset

In [16]:
import kamodo_ccmc.flythrough.model_wrapper as MW
file_dir = "C:/Users/mageb/OneDrive/Documents/Python_Project/BSA_REU2026/data/CTIPe/3523_Katelynn_Greer_042326_IT_1_raw/out"
reader = MW.Model_Reader('CTIPe')

In [17]:
# Prints documentation for model reader
reader.__doc__.split('\n')

['CTIPe model data reader.',
 '',
 '        Inputs:',
 '            file_dir: a string representing the file directory of the',
 '                model output data.',
 "                Note: This reader 'walks' the entire dataset in the directory.",
 '            variables_requested = a list of variable name strings chosen from',
 '                the model_varnames dictionary in this script, specifically the',
 '                first item in the list associated with a given key.',
 '                - If empty, the reader functionalizes all possible variables',
 '                    (default)',
 "                - If 'all', the reader returns the model_varnames dictionary",
 '                    above for only the variables present in the given files.',
 '            filetime = boolean (default = False)',
 '                - If False, the script fully executes.',
 '                - If True, the script only executes far enough to determine the',
 '                    time values associ

In [18]:
# Default functionalization method
# TIEGCM will have a data type error
kamodo_object_default = reader(file_dir) # (gridded_int=True is default)
kamodo_object_default

IndexError: list index out of range

In [28]:
from kamodo_ccmc.tools.functionalize import Functionalize_Dataset
from kamodo import Kamodo

# 1. Define clean coordinate names and standard units for the math typesetter
coord_dict = {
    "time": {"units": "hr", "data": ds["time"].values},
    "plev": {"units": "hPa", "data": ds["plev"].values},
    "lat": {"units": "deg", "data": ds["lat"].values},
    "lon": {"units": "deg", "data": ds["lon"].values},
}

# 2. Add your variables with scientific notation units
var_dict = {
    "density": {"units": "kg/m^3", "data": ds["density"].values},
    "temperature": {"units": "K", "data": ds["temperature"].values},
    "height": {"units": "km", "data": ds["height"].values},
}

# 3. Create and functionalize the container
kamodo_ctipe = Kamodo()
kamodo_ctipe = Functionalize_Dataset(coord_dict, var_dict, kamodo_ctipe)

# 4. Display the LaTeX equations (Leave off print() or any method calls!)
kamodo_ctipe

{density(time, plev, lat, lon): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F3858FE0>, density: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F3858FE0>, temperature(time, plev, lat, lon): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F38AF7E0>, temperature: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F38AF7E0>, height(time, plev, lat, lon): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F38AEAC0>, height: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F38AEAC0>}

In [40]:
coord_dict = {
    "time": {"units": "hr", "data": ds["time"].values},
    "plev": {"units": "hPa", "data": ds["plev"].values},
    "lat": {"units": "deg", "data": ds["lat"].values},
    "lon": {"units": "deg", "data": ds["lon"].values},
}

# Single variable dictionary
var_multiple = {
    "density": {"units": "kg/m^3", "data": ds["density"].values},
    "temperature": {"units": "K", "data": ds["temperature"].values},
    "height": {"units": "km", "data": ds["height"].values},
}
kamodo_gridded = Kamodo()
kamodo_gridded = Functionalize_Dataset(coord_dict, var_multiple, kamodo_gridded)
kamodo_gridded

{density(time, plev, lat, lon): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F66E0900>, density: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F66E0900>, temperature(time, plev, lat, lon): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F66E1120>, temperature: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F66E1120>, height(time, plev, lat, lon): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F66E19E0>, height: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x00000210F66E19E0>}

## Satellite Trajectories

In [44]:
# Description
from kamodo_ccmc.flythrough import SatelliteFlythrough as SF
help(SF.SatelliteTrajectory)

Help on function SatelliteTrajectory in module kamodo_ccmc.flythrough.SatelliteFlythrough:

SatelliteTrajectory(dataset, start_ts, stop_ts, coord_type='GEO', verbose=False)
    Retrieve and return satellite trajectory from HAPI/CDAWeb
    Parameters:
    ----------
    dataset: name of the satellite data set to pull trajectory from
    start_ts: utc timestamp for start of desired time interval
    stop_ts: utc timestamp for end of desired time interval
    coord_type: Pick from GEO, GSM, GSE, or SM
    verbose: Set to true to be overwhelmed with information.
    
    Returns a dictionary with keys: sat_time, c1, c2, and c3.
        sat_time is an array in UTC seconds since 1970-01-01.
        (c1,c2,c3) = (lon, lat, alt) in (deg,deg,km) in the 'GDZ', 'sph'
        coordinate system in SpacePy. See
        kamodo_ccmc.flythrough.utils.ConvertCoord for more info on the
        coordinate systems.
    
    Coordinates are retrieved on a cartesian grid.
    See kamodo_ccmc.flythrough.utils

In [45]:
# Retrieve a real satellite trajectory.
# Typical coordinates possible through SSCWeb are GEO, GSE, SM, and GSM (all cartesian and in R_E).
from datetime import datetime, timezone
start = datetime(2015, 3, 18, 6, 26, 40).replace(tzinfo=timezone.utc)
end = datetime(2015, 3, 18, 12, 11, 40).replace(tzinfo=timezone.utc)
traj_dict, coord_type = SF.SatelliteTrajectory('grace1', start.timestamp(), end.timestamp(), coord_type='GEO')
# Show the structure of the returned dictionary and range of each coordinate.
print(traj_dict.keys(), coord_type)
for key in traj_dict.keys():
    print(type(traj_dict[key]), traj_dict[key].shape, traj_dict[key].min(), traj_dict[key].max())

HAPIError: HAPI error 1402: error in time.min


In [47]:
# Another way to get a trajectory is via the SampleTrajectory flythrough function
help(SF.SampleTrajectory)

Help on function SampleTrajectory in module kamodo_ccmc.flythrough.SatelliteFlythrough:

SampleTrajectory(start_time, stop_time, max_lat=65.0, min_lat=-65.0, lon_perorbit=363.0, max_height=450.0, min_height=400.0, p=0.01, n=2.0)
    Given start and stop times in timestamp form, return a test satellite
    trajectory.
    Parameters:
    ----------
        start_time: utc timestamp in seconds for start
        stop_time: utc timestamp in seconds for stop
        max_lat: maximum latitude for sample trajectory, in degrees
            (default=65.)
        min_lat: minimum latitude for sample trajectory, in degrees
            (default=-65.)
        lon_perorbit: the degrees of longitude per about 90 minute orbit
            (set less than 360 for precession forward in longitude, set less
            than 360 for precession backwards) (default=363.)
        max_height: maximum starting height of orbit in km (default=450.)
        min_height: minimum starting height of orbit in km (default

In [50]:
# Create a custom sample_trajectory

sample_traj, sample_coord = SF.SampleTrajectory(
    start.timestamp(), end.timestamp(), max_height=450., min_height=400., p=0.5, n=30
)
print(sample_traj.keys(), sample_coord)
for key in sample_traj.keys():
    print(type(sample_traj[key]), sample_traj[key].shape, sample_traj[key].min(), sample_traj[key].max())

Attribute/Key names of return dictionary: dict_keys(['sat_time', 'c1', 'c2', 'c3'])
(c1,c2,c3) = (lon, lat, alt) in (deg,deg,km) in the GDZ, sph coordinate system.
sat_time contains the utc timestamps.
dict_keys(['sat_time', 'c1', 'c2', 'c3']) GDZ-sph
<class 'numpy.ndarray'> (690,) 1426660000.0 1426680700.0
<class 'numpy.ndarray'> (690,) -180.0 179.48766328011612
<class 'numpy.ndarray'> (690,) -64.98998926658311 65.0
<class 'numpy.ndarray'> (690,) 202.90125109731187 438.3904760434719


In [51]:
# Choosing a trajectory from TLE (two-line) elements
help(SF.TLETrajectory)

Help on function TLETrajectory in module kamodo_ccmc.flythrough.SatelliteFlythrough:

TLETrajectory(tle_file, start_utcts, stop_utcts, time_cadence, method='forward', verbose=False)
    Use sgp4 to calculate a satellite trajectory given TLEs.
    Parameters:
        tle_file: The file name, including complete file path, of a file
            containing two-line elements. It is assumed that the file has no
            header and no other content.
        start_utcts: The UTC timestamp corresponding to the desired start time.
            Should be an integer.
        stop_utcts: The UTC timestamp corresponding to the desired stop time.
            Should be an integer.
        time_cadence: The number of seconds desired between trajectory
            positions. Should be an integer.
        method: 'forward' or 'nearest'. This keyword changes the propagation
            method for timestamps between TLEs, not for timestamps before the
            first TLE or after the last TLE. The 'for

In [ ]:
# How to get TLEs from files
tle_file = ''
time_cadence = 60 # Seconds between propagated trajectory positions
start_utcts = datetime(2015, 3, 18, 0, 20).replace(tzinfo=timezone.utc).timestamp()

end_utcts = datetime(2015, 3, 21, 0, 0).replace(tzinfo=timezone.utc).timestamp()
tle_dict, coord_type = SF.TLETrajectory(tle_file, start_utcts, end_utcts, time_cadence)
# Show the structure of the returned dictionary and range of each coordinate.
print(tle_dict.keys(), coord_type)
for key in tle_dict.keys():
    print(type(tle_dict[key]), tle_dict[key].shape, tle_dict[key].min(), tle_dict[key].max())

In [53]:
# Load trajectory from a file

help(SF.O.SF_read)

Help on function SF_read in module kamodo_ccmc.flythrough.SF_output:

SF_read(filename)
    Collect input function calls into one function.
    
    filename = string with complete filepath. The file extension must be one
        of 'nc'for a netCDF4 file, 'csv' for a comma separated file, or 'txt'
        for a tab separated file.
    
    Output: a nested dictionary containing the metadata, data, and units.



In [ ]:
file_data = SF.O.SF_read('')
coord_sys = file_data['metadata']['coord_type'] + '-' + file_data['metadata']['coord_grid']
# Show the structure of the returned dictionary.
print(file_data.keys(), coord_sys)
for key in file_data.keys():
    print(file_data[key].keys())

In [54]:
# Write trajectory to a file
help(SF.O.SF_write)

Help on function SF_write in module kamodo_ccmc.flythrough.SF_output:

SF_write(filename, model_filename, model_name, results_dict, results_units, coord_sys)
    Collect output function calls into one function.
    
    Inputs:
        filename = string with complete filepath. The file extension must be
            one of 'nc' for a netCDF4 file, 'csv' for a comma separated file,
            or 'txt' for a tab separated file.
        model_filename = A list of the model data filenames or prefixes used
            to generate the data. Filenames should include the full file path.
        model_name = A string indicating the model name.
        results_dict = A dictionary with variable names as keys (strings) and
            the time series data as the values (one array per key).
        results_units = A dictionary with variable names as keys (strings) and
            the units as the values (one value per key).
        coord_sys = one of 'GDZ', 'GEO', 'GSM', 'GSE', 'SM', 'GEI', 'MAG',


In [ ]:
# Write TLE trajectory to a new file
tle_units = {
    'sat_time', 's', 'c1', 'R_E', 'c2':'R_E', 'c3', 'R_E'
}
SF.O.SF_write(
    '.../trajectories', '', '', tle_dict, tle_units, 'teme-car'
)

## Coordinate Conversions

In [55]:
# import sat flythrough coords conversions code
from kamodo_ccmc.flythrough.utils import ConvertCoord
help(ConvertCoord)

Help on function ConvertCoord in module kamodo_ccmc.flythrough.utils:

ConvertCoord(inTime, c1, c2, c3, inCoord, inType, outCoord, outType, verbose=False)
    This function uses spacepy and astropy to convert time and position arrays
    from one coordinate system to another. It will correct obvious errors in
    the return units, but may not catch all incorrect values.
    
    INPUTS:
    inTime array:  time in UTC timestamp
    c1 array:  x (in R_earth)*, lon (in deg)
    c2 array:  y (in R_earth)*, lat (in deg)
    c3 array:  z (in R_earth)*, alt (in km), radius (in R_earth)
    inCoord string: case-sensitive string from list:
        'GDZ', 'GEO', 'GSM', 'GSE', 'SM', 'GEI', 'MAG', 'SPH', 'RLL'
        (SpacePy coordinates)
        'teme', 'icrs', 'fk5', 'fk4', 'itrs', 'galactic', 'galactocentric',
        'cirs', 'tete', 'precessedgeocentric', 'geocentricmeanecliptic',
        'geocentrictrueecliptic', 'hcrs', 'barycentricmeanecliptic',
        'heliocentricmeanecliptic', 'barycen

In [61]:
# Retrieve real sat traj as input

# Typical coordinates possible through SSCWeb are GEO, GSE, SM, and GSM (all cartesian and in R_E).
from kamodo_ccmc.flythrough import SatelliteFlythrough as SF
from datetime import datetime, timezone

start = datetime(2015, 3, 18, 6, 26, 40).replace(tzinfo=timezone.utc)
end = datetime(2015, 3, 18, 12, 11, 40).replace(tzinfo=timezone.utc)

traj_dict, coord_type = SF.SatelliteTrajectory('grace1', start.timestamp(), end.timestamp(), coord_type='GEO')
# Show the range of each coordinate. 
print('Coordinate system:', coord_type)
print('sat_time = utc timestamp:', traj_dict['sat_time'].min(), traj_dict['sat_time'].max())
print(datetime.utcfromtimestamp(traj_dict['sat_time'].min()), datetime.utcfromtimestamp(traj_dict['sat_time'].max()))
print('c1 = x(R_E):', traj_dict['c1'].min(), traj_dict['c1'].max())
print('c2 = y(R_E):', traj_dict['c2'].min(), traj_dict['c1'].max())
print('c3 = z(R_E):', traj_dict['c3'].min(), traj_dict['c1'].max())

HAPIError: HAPI error 1402: error in time.min


In [64]:
from datetime import datetime, timezone
import numpy as np
import requests

# 1. Standardize dates to ISO strings for SSCWeb
start_str = "2015-03-18T06:26:40Z"
end_str   = "2015-03-18T12:11:40Z"

t_start = datetime(2015, 3, 18, 6, 26, 40, tzinfo=timezone.utc).timestamp()
t_stop  = datetime(2015, 3, 18, 12, 11, 40, tzinfo=timezone.utc).timestamp()

# UPDATE: Use the active SSCWeb location endpoint query structure
url = "https://sscweb.gsfc.nasa.gov/WS/ssce/2/locations"

payload = {
    "satellite": ["grace1"],
    "beginTime": start_str.replace("-", "").replace(":", ""), # Cleans to 20150318T062640Z
    "endTime": end_str.replace("-", "").replace(":", ""),
    "coordinateSystem": "Geo"
}
headers = {'Content-Type': 'application/json', 'Accept': 'application/json'}

print("Querying active SSCWeb REST server...")
response = requests.post(url, json=payload, headers=headers)

if response.status_code == 200:
    res_data = response.json()
    
    # Extract the geographic coordinates (returned in Earth Radii)
    coords = res_data['Result']['Data'][1][0]['Coordinates'][1][0]
    x_pts = np.array(coords['X'])
    y_pts = np.array(coords['Y'])
    z_pts = np.array(coords['Z'])
    
    # Line up the timestamp track matching the point array footprint
    sat_time = np.linspace(t_start, t_stop, len(x_pts))
    
    # Structure the dictionary identically to Kamodo's native outputs
    traj_dict = {
        'sat_time': sat_time,
        'c1': x_pts,
        'c2': y_pts,
        'c3': z_pts
    }
    coord_type = 'GEO'
    
    print('\nCoordinate system:', coord_type)
    print('sat_time = utc timestamp:', traj_dict['sat_time'].min(), traj_dict['sat_time'].max())
    print(datetime.fromtimestamp(traj_dict['sat_time'].min(), tz=timezone.utc), 
          datetime.fromtimestamp(traj_dict['sat_time'].max(), tz=timezone.utc))
    print('c1 = x(R_E):', traj_dict['c1'].min(), traj_dict['c1'].max())
    print('c2 = y(R_E):', traj_dict['c2'].min(), traj_dict['c2'].max())
    print('c3 = z(R_E):', traj_dict['c3'].min(), traj_dict['c3'].max())
else:
    print(f"Server Error: {response.status_code}")

Querying active SSCWeb REST server...
Server Error: 404


## Performing a Flythrough

In [66]:
#Import
from kamodo_ccmc.flythrough import SatelliteFlythrough as SF
help(SF.RealFlight)

Help on function RealFlight in module kamodo_ccmc.flythrough.SatelliteFlythrough:

RealFlight(dataset, start, stop, model, file_dir, variable_list, coord_type='GEO', output_name='', plot_coord='GEO', verbose=False)
    Retrieves the trajectory for the satellite requested and then flies that
    trajectory through the model data requested.
    
    dataset: name of the satellite data set to pull trajectory from
    start: utc timestamp for start of desired time interval
    stop: utc timestamp for end of desired time interval
    model: 'CTIPe','IRI', ...
    file_dir: complete path to where model data files are stored
    variable_list: List of standardized variable names. See model variable
        output for details. Dimensions must be at least time + 2D spatial.
    coord_type: Pick from GEO, GSM, GSE, or SM for the satellite trajectory.
    output_name: complete path with filename (with the extension) for the file
        to write the results to. Plotting filenames are determined b

In [71]:


# Select input values
import datetime as dt

# Choose input values for RealFlight function call.
model = 'CTIPe'  # Choose the model.
# Full file path to model output data.
file_dir = "C:/Users/mageb/OneDrive/Documents/Python_Project/BSA_REU2026/data/CTIPe/3523_Katelynn_Greer_042326_IT_1_raw/out"

start_utcts = dt.datetime(2015, 3, 17, 0).replace(tzinfo=dt.timezone.utc).timestamp()
end_utcts = dt.datetime(2015, 3, 17, 2).replace(tzinfo=dt.timezone.utc).timestamp()-1
variables = ['rho_n', 'TEC']  # one or more variable names to retrieve
dataset = 'cnofs' 

# Use https://sscweb.gsfc.nasa.gov/ to find the satellite name and time range desired.
coord_type = 'GEO'  # Desired coordinate system for retrieved trajectory.
# Choose naming convention for output files
output_name = 'C:/Users/mageb/OneDrive/Documents/Python_Project/BSA_REU2026/data/RealFlightExample_CTIPe.txt' 
plot_coord = 'GSE'  # Coordinate system chosen for output plots
# See https://sscweb.gsfc.nasa.gov/users_guide/Appendix_C.shtml for a description of coordinate types

In [72]:
# Remove the previously created output file.
import os
if os.path.isfile(output_name):
    os.remove(output_name)
    print(output_name, 'file removed.')
else:
    print(output_name, 'file not found.')

C:/Users/mageb/OneDrive/Documents/Python_Project/BSA_REU2026/data/RealFlightExample_CTIPe.txt file not found.


In [73]:
# Run RealFlight flythrough command. 
results_real = SF.RealFlight(dataset, start_utcts, end_utcts, model, file_dir, variables, 
                             coord_type=coord_type, output_name=output_name, plot_coord=plot_coord)
# The numerous 'Time slice index' messages printed are generated by the lazy time interpolation,
# which only loads the time slices necessary to perform the flythrough execution requested.
# Open plots in separate internet browser window for interactivity. Nothing will open here.

HAPIError: HAPI error 1402: error in time.min


In [74]:
# Run FakeFlight. Note the same variables defined above are used here. 
results_fake = SF.FakeFlight(start_utcts, end_utcts, model, file_dir, variables, max_height=400, min_height=300, n=20)

Attribute/Key names of return dictionary: dict_keys(['sat_time', 'c1', 'c2', 'c3'])
(c1,c2,c3) = (lon, lat, alt) in (deg,deg,km) in the GDZ, sph coordinate system.
sat_time contains the utc timestamps.


IndexError: list index out of range

In [75]:
help(SF.TLEFlight)

Help on function TLEFlight in module kamodo_ccmc.flythrough.SatelliteFlythrough:

TLEFlight(tle_file, start, stop, time_cadence, model, file_dir, variable_list, output_name='', plot_coord='GEO', method='forward', verbose=False)
    Use sgp4 to calculate a satellite trajectory given TLEs, then fly the
    trajectory through the chosen model data. If the time cadence does not
    evenly divide into the range of timestamps given, then the ending time
    value will be extended so that the entire requested range will be covered.
    Parameters:
        tle_file: The file name, including complete file path, of a file
            containing two-line elements. It is assumed that the file has no
            header and no other content.
        start_utcts: The UTC timestamp corresponding to the desired start time.
            Should be an integer.
        stop_utcts: The UTC timestamp corresponding to the desired stop time.
            Should be an integer.
        time_cadence: The number of 

In [ ]:
# Run TELflight via flythrough
tle_file = 'C:/Users/mageb/OneDrive/Documents/Python_Project/BSA_REU2026/data/Trajectories/grace1_tle.txt'
time_cadence = 60. # seconds between propagated trajectory positions
results_tle = SF.TLEFlight(tle_file, start_utcts, end_utcts, time_cadence, model, file_dir, variables)

## Constellation Planning

In [88]:
from kamodo_ccmc.flythrough import model_wrapper as MW 
import datetime as dt

model, file_dir = 'CTIPe', "C:/Users/mageb/OneDrive/Documents/Python_Project/BSA_REU2026/data/CTIPe/3523_Katelynn_Greer_042326_IT_1_raw/out"
start_utcts = dt.datetime(2015, 3, 17, 0).replace(tzinfo=dt.timezone.utc).timestamp()
end_utcts = dt.datetime(2015, 3, 17, 6).replace(tzinfo=dt.timezone.utc).timestamp()-1
# Performs any data prep needed

In [89]:
# Import function to retrieve the grace1 trajectory from the SSCWeb.
from kamodo_ccmc.flythrough import SatelliteFlythrough as SF
# Typical coordinates possible through SSCWeb are GEO, GSE, SM, and GSM (all cartesian and in R_E).
input_coord = 'GEO'
traj_dict, coord_type = SF.SatelliteTrajectory('grace1', start_utcts, end_utcts, coord_type=input_coord)

HAPIError: HAPI error 1402: error in time.min
